## Ingest Dimension Data into Bronze Layer

In [0]:
%sql

use catalog olist_ecommerce_project;
use schema bronze;

In [0]:
catalog_name = 'olist_ecommerce_project'

In [0]:
raw_path = "/Volumes/olist_ecommerce_project/raw/dataset";

### Libraries Import

In [0]:
# Importing libraries
from pyspark.sql.functions import current_timestamp, lit , to_timestamp

### Customer Table

In [0]:


# 1. Read raw CSV
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)   # safe for now, we'll cast explicitly next
    .csv(f"{raw_path}/olist_customers_dataset.csv")
)

# 2. Light, safe type casting
df_customers = (
    df_customers
    .withColumn("customer_zip_code_prefix", df_customers["customer_zip_code_prefix"].cast("string"))
    # customer_id, customer_unique_id, customer_city, customer_state are already strings
)

# 3. Add audit/metadata columns
df_customers = (
    df_customers
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_customers_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_customers.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("olist_ecommerce_project.bronze.brz_customers")
)

In [0]:
df_customers.show(10)

### GeoLocation Table

In [0]:
# 1. Read raw CSV
df_geolocation = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_geolocation_dataset.csv")
)

# 2. Light, safe type casting
df_geolocation = (
    df_geolocation
    .withColumn("geolocation_zip_code_prefix", df_geolocation["geolocation_zip_code_prefix"].cast("string"))
    .withColumn("geolocation_lat", df_geolocation["geolocation_lat"].cast("double"))
    .withColumn("geolocation_lng", df_geolocation["geolocation_lng"].cast("double"))
    # geolocation_city, geolocation_state are already strings
)

# 3. Add audit/metadata columns
df_geolocation = (
    df_geolocation
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_geolocation_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_geolocation.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_geolocation")
)

In [0]:
display(df_geolocation.limit(20))

### Product Table

In [0]:
# 1. Read raw CSV
df_products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_products_dataset.csv")
)

# 2. Cast types FIRST (using original typo'd names, since that's what the CSV has)
df_products = (
    df_products
    .withColumn("product_name_lenght", df_products["product_name_lenght"].cast("integer"))
    .withColumn("product_description_lenght", df_products["product_description_lenght"].cast("integer"))
    .withColumn("product_photos_qty", df_products["product_photos_qty"].cast("integer"))
    .withColumn("product_weight_g", df_products["product_weight_g"].cast("double"))
    .withColumn("product_length_cm", df_products["product_length_cm"].cast("double"))
    .withColumn("product_height_cm", df_products["product_height_cm"].cast("double"))
    .withColumn("product_width_cm", df_products["product_width_cm"].cast("double"))
)

# 3. THEN rename the typo'd columns
df_products = (
    df_products
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
)

# 4. Add audit/metadata columns
df_products = (
    df_products
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_products_dataset.csv"))
)

# 5. Write as Delta table into Bronze schema (overwrite to fix earlier run)
(
    df_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_products")
)

In [0]:
display(df_products.limit(20))

###Seller Table

In [0]:
# 1. Read raw CSV
df_sellers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_sellers_dataset.csv")
)

# 2. Light, safe type casting
df_sellers = (
    df_sellers
    .withColumn("seller_zip_code_prefix", df_sellers["seller_zip_code_prefix"].cast("string"))
    # seller_id, seller_city, seller_state are already strings
)

# 3. Add audit/metadata columns
df_sellers = (
    df_sellers
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_sellers_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_sellers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_sellers")
)

In [0]:
display(df_sellers.limit(20))

### Category Name Table

In [0]:
# 1. Read raw CSV
df_category_translation = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/product_category_name_translation.csv")
)

# 2. Light, safe type casting
# Both columns are already strings - no casting needed here

# 3. Add audit/metadata columns
df_category_translation = (
    df_category_translation
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("product_category_name_translation.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_category_translation.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_category_name")
)

In [0]:
display(df_category_translation.limit(20))

### Orders Table

In [0]:
# 1. Read raw CSV
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_orders_dataset.csv")
)

# 2. Light, safe type casting — cast all date-looking columns to timestamp
df_orders = (
    df_orders
    .withColumn("order_purchase_timestamp", df_orders["order_purchase_timestamp"].cast("timestamp"))
    .withColumn("order_approved_at", df_orders["order_approved_at"].cast("timestamp"))
    .withColumn("order_delivered_carrier_date", df_orders["order_delivered_carrier_date"].cast("timestamp"))
    .withColumn("order_delivered_customer_date", df_orders["order_delivered_customer_date"].cast("timestamp"))
    .withColumn("order_estimated_delivery_date", df_orders["order_estimated_delivery_date"].cast("timestamp"))
    # order_id, customer_id, order_status are already strings
)

# 3. Add audit/metadata columns
df_orders = (
    df_orders
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_orders_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_orders")
)

In [0]:
display(df_orders.limit(20))

### Order Items Table

In [0]:


# 1. Read raw CSV
df_order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_order_items_dataset.csv")
)

# 2. Light, safe type casting
df_order_items = (
    df_order_items
    .withColumn("order_item_id", df_order_items["order_item_id"].cast("integer"))
    .withColumn("shipping_limit_date", to_timestamp(df_order_items["shipping_limit_date"], "M/d/yyyy H:mm"))
    .withColumn("price", df_order_items["price"].cast("double"))
    .withColumn("freight_value", df_order_items["freight_value"].cast("double"))
)

# 3. Add audit/metadata columns
df_order_items = (
    df_order_items
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_order_items_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_order_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")
)

In [0]:
display(df_order_items.limit(20))

### Order Payments Table

In [0]:
# 1. Read raw CSV
df_payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_path}/olist_order_payments_dataset.csv")
)

# 2. Light, safe type casting
df_payments = (
    df_payments
    .withColumn("payment_sequential", df_payments["payment_sequential"].cast("integer"))
    .withColumn("payment_installments", df_payments["payment_installments"].cast("integer"))
    .withColumn("payment_value", df_payments["payment_value"].cast("double"))
    # order_id, payment_type are already strings
)

# 3. Add audit/metadata columns
df_payments = (
    df_payments
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_order_payments_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_payments")
)

In [0]:
display(df_payments.limit(20))

### Order Reviews Table

In [0]:
# 1. Read raw CSV — with proper multiline/quote handling for messy review text
df_reviews = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)      # allows fields to span multiple lines
    .option("quote", '"')           # tells Spark a quoted field can contain commas/newlines
    .option("escape", '"')          # handles escaped quotes inside quoted text
    .csv(f"{raw_path}/olist_order_reviews_dataset.csv")
)

In [0]:
display(df_reviews.limit(20))

In [0]:
df_reviews.select("review_id", "review_score", "review_creation_date").show(20, truncate=False)

In [0]:
# 2. Light, safe type casting
df_reviews = (
    df_reviews
    .withColumn("review_score", df_reviews["review_score"].cast("integer"))
    .withColumn("review_creation_date", to_timestamp(df_reviews["review_creation_date"], "M/d/yyyy H:mm"))
    .withColumn("review_answer_timestamp", to_timestamp(df_reviews["review_answer_timestamp"], "M/d/yyyy H:mm"))
)

# 3. Add audit/metadata columns
df_reviews = (
    df_reviews
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("olist_order_reviews_dataset.csv"))
)

# 4. Write as Delta table into Bronze schema
(
    df_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.bronze.brz_reviews")
)